# Distribution VAE — Quickstart

This notebook walks through the basic workflow:
1. Train on synthetic data
2. Inspect reconstructions
3. Explore the latent space
4. Encode distributions to latent vectors

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np

from dist_vae.model import DistributionVAE
from dist_vae.data import SyntheticDistributionDataset
from dist_vae.train import Trainer

## 1. Create Synthetic Data

Each distribution is a random mixture of 1-4 Gaussians, converted to a quantile grid.

In [ ]:
grid_size = 256
train_dataset = SyntheticDistributionDataset(n_distributions=1000, grid_size=grid_size, seed=42)
val_dataset = SyntheticDistributionDataset(n_distributions=200, grid_size=grid_size, seed=123)
print(f"Train: {len(train_dataset)} distributions, Val: {len(val_dataset)} distributions")

# Visualize a few distributions
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
x = np.linspace(0, 1, grid_size)
for i, ax in enumerate(axes):
    grid, _, _ = train_dataset[i]
    ax.plot(x, grid.numpy())
    ax.set_title(f'Distribution {i}')
    ax.set_xlabel('Quantile')
    ax.set_ylabel('Value')
plt.tight_layout()
plt.show()

## 2. Create and Train Model

In [ ]:
model = DistributionVAE(grid_size=grid_size, latent_dim=32, hidden_dim=128, beta=0.01)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")

In [ ]:
config = {
    'training': {'epochs': 50, 'batch_size': 64, 'lr': 1e-3, 'weight_decay': 1e-4,
                 'grad_clip': 1.0, 'val_fraction': 0.1, 'seed': 42},
    'model': {'beta': 0.01, 'beta_warmup_epochs': 5},
    'loss': {'cramer': 1.0, 'wasserstein1': 0.0, 'kl_divergence': 0.0},
    'logging': {'wandb': False, 'print_every': 10, 'checkpoint_dir': 'checkpoints/'},
}
trainer = Trainer(model, train_dataset, val_dataset, config)
history = trainer.train()

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'], label='Val')
axes[0].set_title('Total Loss')
axes[0].legend()
axes[0].set_xlabel('Epoch')

axes[1].plot(history['train_recon'], label='Train')
axes[1].plot(history['val_recon'], label='Val')
axes[1].set_title('Reconstruction Loss')
axes[1].legend()
axes[1].set_xlabel('Epoch')

axes[2].plot(history['beta'], label='Beta')
axes[2].set_title('KL Weight (Beta)')
axes[2].set_xlabel('Epoch')
plt.tight_layout()
plt.show()

## 3. Inspect Reconstructions

In [ ]:
from dist_vae.eval import plot_reconstructions, evaluate_reconstruction

metrics = evaluate_reconstruction(model, val_dataset)
print("Reconstruction metrics:")
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.6f}")

plot_reconstructions(model, val_dataset, n_examples=9)

## 4. Explore Latent Space

In [ ]:
from dist_vae.eval import plot_latent_space, plot_interpolations, plot_latent_statistics

plot_latent_space(model, val_dataset, method='pca')

In [ ]:
plot_interpolations(model, val_dataset, idx_pairs=[(0, 10), (5, 15)], n_steps=8)

In [ ]:
plot_latent_statistics(model, val_dataset)

## 5. Encode Distributions to Latent Vectors

In [ ]:
# Encode individual distributions
model.eval()
with torch.no_grad():
    grid, _, _ = val_dataset[0]
    mu = model.encode_distribution(grid.unsqueeze(0))
    print(f"Latent vector shape: {mu.shape}")
    print(f"First 8 dims: {mu[0, :8].numpy()}")

# Encode from raw samples
raw_samples = torch.randn(100)  # 100 samples from N(0,1)
with torch.no_grad():
    mu = model.encode_distribution(raw_samples.unsqueeze(0))
    print(f"\nEncoded N(0,1) samples -> latent: {mu[0, :8].numpy()}")